### Bond valence site energy method for studying ionic transport properties

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dembart/gridmlip/blob/main/notebooks/05_lfp_lco.ipynb)

Computed with [gridmlip](https://github.com/dembart/gridmlip).

Bond valence site energy (BVSE) maps of olivine LiFePO4 and layered LiCoO2, their percolation barriers, and NEB starting paths of the vacancy-assisted Li hops. Structures: experimental phases from the [Materials Project](https://materialsproject.org) (CC BY 4.0).

### Setup

In [ ]:
!pip install -q git+https://github.com/dembart/gridmlip.git

In [ ]:
!mkdir -p data
!wget -q -O data/LiFePO4_mp-aaaabcdl.cif https://raw.githubusercontent.com/dembart/gridmlip/main/notebooks/data/LiFePO4_mp-aaaabcdl.cif
!wget -q -O data/LiCoO2_mp-aaaabhik.cif https://raw.githubusercontent.com/dembart/gridmlip/main/notebooks/data/LiCoO2_mp-aaaabhik.cif

In [ ]:
from ase.io import read
import gridmlip as gm

lfp = read("data/LiFePO4_mp-aaaabcdl.cif")

In [ ]:
gm.view.structure_view(lfp)

#### BVSE map

The Li+ energy on a 0.2 Å mesh with all Li removed from the host; percolation barriers are the lowest levels at which the accessible region connects in 1, 2 and 3 dimensions.

In [ ]:
lfp_map = gm.bvse(lfp, "Li+", resolution=0.2, verbose=False)

LiFePO4: 1D channels along b. Drag the level slider or use the 1D/2D/3D presets.

In [ ]:
lfp_map.view(
    #polyhedra=["Fe", "P"],
    #hide_bonds = ['Li-O']
    )

In [ ]:
lfp_map.write("bvse_map_lfp.grd")
# VESTA 3: open data/LiFePO4_mp-aaaabcdl.cif, then Edit > Edit Data > Volumetric Data > Import... bvse_map_lfp.grd

### Try the same for LiCoO2

In [ ]:
lco = read("data/LiCoO2_mp-aaaabhik.cif")

### your code ###

#### Vacancy-assisted hops

Hops between Voronoi neighbours of the Li sublattice, one per symmetry class. Each hop is sampled in a supercell with the hopping Li removed and a vacancy on the target site, only in a tube around the hop.

In [ ]:
lfp_hops = gm.Hops.from_voronoi(lfp, "Li", resolution=0.2)
lfp_hops.summary()

In [ ]:
lfp_maps = lfp_hops.evaluate(gm.BVSE("Li+", verbose=False))
print("hop barriers  ", lfp_maps.barriers.round(3))
print("network       ", lfp_maps.percolation_barriers())

The lowest hop, with its NEB images as a trajectory over the isosurface (each frame shows its map energy).

In [ ]:
hmap = lfp_maps[int(lfp_maps.barriers.argmin())]
path = hmap.path(n_images=9)
print(path)
hmap.view(path=path, polyhedra=False, 
          #hide=["Fe", "P", "O"]
          )

The same for LiCoO2: in-plane hops percolate in 2D, the hops across the CoO2 sheets set the 3D barrier.

In [ ]:
### your code ###

## NEB inputs for VASP

One folder per hop (00/POSCAR, 01/POSCAR, ...) and paths.json with the grid estimates. Keep the images as given in the NEB (no re-interpolation).

In [ ]:
lfp_maps.write_paths("neb_lfp", format="vasp", n_images=9)
!ls neb_lfp neb_lfp/hop_000

With NEB energies (lowest and highest along each hop), the network barriers follow from the same hop network:

```python
gm.network.percolation_barriers(lfp_hops.network, emins[lfp_hops.inverse], emaxs[lfp_hops.inverse])
```